<div align='center'>

# 🌍 Hackathon IndabaX Cameroon 2026
## Starter Notebook — 🇫🇷 Français
### *L'IA au service de la résilience climatique et sanitaire*

---

| Étape | Description |
|-------|-------------|
| **1** | Setup & Chargement des données |
| **2** | Exploration (EDA) |
| **3** | Nettoyage & Feature Engineering |
| **4** | Modélisation Baseline |
| **5** | Pistes pour aller plus loin |

> ⚠️ Ce notebook est un **point de départ**. À vous d'innover !

</div>

---
## 1. ⚙️ Setup & Chargement

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import warnings; warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
SEED = 42

In [ ]:
# ── Chargement du dataset officiel ───────────────────────────────────────────
df = pd.read_excel('../data/Dataset_complet_Meteo.xlsx')

# Conversion des colonnes numériques (stockées en string dans le fichier)
num_cols = [
    'temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean',
    'apparent_temperature_mean', 'precipitation_sum', 'rain_sum',
    'wind_speed_10m_max', 'wind_gusts_10m_max',
    'shortwave_radiation_sum', 'et0_fao_evapotranspiration',
    'sunshine_duration', 'latitude', 'longitude'
]
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f"✅ Dataset : {df.shape[0]:,} obs × {df.shape[1]} variables")
print(f"   Période  : {df['time'].min().date()} → {df['time'].max().date()}")
print(f"   Villes   : {df['city'].nunique()} | Régions : {df['region'].nunique()}")
df.head(3)

---
## 2. 🔍 Exploration des Données (EDA)

In [ ]:
# ── Statistiques descriptives ─────────────────────────────────────────────────
df[['temperature_2m_mean', 'precipitation_sum', 'wind_speed_10m_max',
    'shortwave_radiation_sum', 'et0_fao_evapotranspiration']].describe().round(2)

In [ ]:
# ── Saisonnalité nationale ────────────────────────────────────────────────────
df['month'] = df['time'].dt.month
monthly = df.groupby('month').agg(
    temp_moy=('temperature_2m_mean', 'mean'),
    pluie_moy=('precipitation_sum', 'mean')
)

fig, ax1 = plt.subplots(figsize=(11, 4))
mois = ['Jan','Fév','Mar','Avr','Mai','Juin','Juil','Aoû','Sep','Oct','Nov','Déc']
ax1.bar(mois, monthly['pluie_moy'], color='steelblue', alpha=0.6, label='Précipitations (mm)')
ax2 = ax1.twinx()
ax2.plot(mois, monthly['temp_moy'], color='tomato', marker='o', lw=2, label='Température (°C)')
ax1.set_title('Saisonnalité — Température & Précipitations', fontweight='bold')
ax1.set_ylabel('Précipitations (mm)'); ax2.set_ylabel('Température (°C)')
plt.tight_layout(); plt.show()

In [ ]:
# ── Carte interactive des 42 villes ───────────────────────────────────────────
city_stats = df.groupby(['city', 'region', 'latitude', 'longitude']).agg(
    temp_moy=('temperature_2m_mean', 'mean'),
    radiation=('shortwave_radiation_sum', 'mean')
).reset_index().round(2)

fig = px.scatter_mapbox(
    city_stats, lat='latitude', lon='longitude',
    color='temp_moy', size='radiation', hover_name='city',
    hover_data={'region': True}, color_continuous_scale='RdYlBu_r',
    zoom=5, mapbox_style='open-street-map',
    title='Carte des 42 Villes — Température & Rayonnement Moyens'
)
fig.update_layout(height=500)
fig.show()

In [ ]:
# ── Corrélations entre variables météo ────────────────────────────────────────
corr_cols = ['temperature_2m_mean', 'precipitation_sum', 'wind_speed_10m_max',
             'shortwave_radiation_sum', 'et0_fao_evapotranspiration', 'sunshine_duration']

plt.figure(figsize=(8, 6))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Matrice de Corrélation', fontweight='bold')
plt.tight_layout(); plt.show()

---
## 3. 🔧 Nettoyage & Feature Engineering

In [ ]:
# ── Variables temporelles ─────────────────────────────────────────────────────
df['year']         = df['time'].dt.year
df['quarter']      = df['time'].dt.quarter
df['day_of_year']  = df['time'].dt.dayofyear
df['month_sin']    = np.sin(2 * np.pi * df['month'] / 12)   # encodage cyclique
df['month_cos']    = np.cos(2 * np.pi * df['month'] / 12)
df['is_dry_season']= df['month'].isin([11,12,1,2,3]).astype(int)

# ── Variables dérivées (indicateurs de pollution potentielle) ─────────────────
df['temp_amplitude']   = df['temperature_2m_max'] - df['temperature_2m_min']
df['sunshine_ratio']   = df['sunshine_duration'] / (df['daylight_duration'] + 1e-6)
df['is_no_wind']       = (df['wind_speed_10m_max'] < 5).astype(int)  # stagnation
df['is_no_rain']       = (df['precipitation_sum'] < 0.1).astype(int)  # pas de lessivage

# ── Variables de lag (séries temporelles) ─────────────────────────────────────
df = df.sort_values(['city', 'time']).reset_index(drop=True)
for lag in [1, 3, 7]:
    df[f'temp_lag{lag}'] = df.groupby('city')['temperature_2m_mean'].shift(lag)
    df[f'wind_lag{lag}'] = df.groupby('city')['wind_speed_10m_max'].shift(lag)

df['temp_roll7'] = df.groupby('city')['temperature_2m_mean'].transform(
    lambda x: x.rolling(7, min_periods=1).mean())

# ── Encodage des catégories ───────────────────────────────────────────────────
df['region_enc'] = df['region'].astype('category').cat.codes
df['city_enc']   = df['city'].astype('category').cat.codes

print(f"✅ Features construites. Nouvelles colonnes : {df.shape[1]} au total")

In [ ]:
# ── Construction du proxy PM2.5 ───────────────────────────────────────────────
#
# En l'absence de mesures directes de qualité de l'air, on construit
# un proxy basé sur les conditions favorisant la pollution :
#   • Température élevée + fort ensoleillement → réactions photochimiques
#   • Absence de vent → stagnation des particules
#   • Absence de pluie → pas d'effet de lessivage
#   • Saison sèche → poussières & feux de brousse
#
# ⚠️ À améliorer avec des données réelles (API Air Quality Open-Meteo)
#    Voir : data/README.md

df['pm25_proxy'] = (
    0.35 * df['temperature_2m_mean'].fillna(df['temperature_2m_mean'].mean())
    + 0.25 * df['shortwave_radiation_sum'].fillna(0)
    + 0.20 * df['et0_fao_evapotranspiration'].fillna(0)
    + 8.0  * df['is_no_wind']
    + 5.0  * df['is_no_rain']
    + 4.0  * df['is_dry_season']
).clip(lower=0)

print("Proxy PM2.5 — aperçu :")
print(df['pm25_proxy'].describe().round(2))

---
## 4. 🤖 Modélisation Baseline

> **Objectif :** Prédire `pm25_proxy` à partir des variables météorologiques.

In [ ]:
# ── Préparation du jeu de données ─────────────────────────────────────────────
FEATURES = [
    'temperature_2m_mean', 'temperature_2m_max', 'temperature_2m_min',
    'precipitation_sum', 'wind_speed_10m_max', 'wind_gusts_10m_max',
    'shortwave_radiation_sum', 'et0_fao_evapotranspiration', 'sunshine_ratio',
    'temp_amplitude', 'is_no_wind', 'is_no_rain', 'is_dry_season',
    'month_sin', 'month_cos', 'day_of_year',
    'temp_lag1', 'temp_lag7', 'wind_lag1', 'temp_roll7',
    'latitude', 'longitude', 'region_enc', 'city_enc'
]
TARGET = 'pm25_proxy'

df_model = df[FEATURES + [TARGET]].dropna()
X, y = df_model[FEATURES], df_model[TARGET]

# Split temporel (80% train, 20% test) — NE PAS utiliser un split aléatoire !
split = int(len(df_model) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

print(f"Train : {len(X_train):,} | Test : {len(X_test):,}")

In [ ]:
# ── Entraînement du modèle ────────────────────────────────────────────────────
model = RandomForestRegressor(n_estimators=100, max_depth=10, n_jobs=-1, random_state=SEED)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"📊 Performances :")
print(f"   R²   = {r2_score(y_test, y_pred):.4f}")
print(f"   MAE  = {mean_absolute_error(y_test, y_pred):.4f}")

In [ ]:
# ── Importance des variables ───────────────────────────────────────────────────
importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 5))
importances.sort_values().plot(kind='barh', color='steelblue')
plt.title('Top 15 — Importance des Variables', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Prédictions vs Réalité ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter
axes[0].scatter(y_test[:1000], y_pred[:1000], alpha=0.3, s=8, color='steelblue')
lim = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lim, lim, 'r--', lw=1.5)
axes[0].set(xlabel='Réel', ylabel='Prédit', title='Réel vs Prédit')

# Résidus
axes[1].hist(y_test.values - y_pred, bins=50, color='salmon', edgecolor='white')
axes[1].axvline(0, color='navy', lw=1.5, ls='--')
axes[1].set(xlabel='Résidu', title='Distribution des Résidus')

plt.suptitle('Diagnostic du Modèle Random Forest', fontweight='bold')
plt.tight_layout(); plt.show()

---
## 5. 🚀 Pistes pour Aller Plus Loin

### 🔬 Améliorer le modèle
```python
# Modèles plus puissants
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# Validation temporelle rigoureuse
from sklearn.model_selection import TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=5)

# Séries temporelles
from prophet import Prophet  # ou LSTM, Temporal Fusion Transformer
```

### 📡 Enrichir les données (⭐ Recommandé)
```python
# API Qualité de l'air Open-Meteo (PM2.5, PM10, poussières...)
# → Voir data/README.md pour le code complet
import openmeteo_requests
om = openmeteo_requests.Client()
# pm2_5, pm10, dust, european_aqi disponibles gratuitement !
```

### 🖥️ Dashboard Streamlit
```python
# dashboard_template/app_template.py — Template fourni dans ce repo
# streamlit run dashboard_template/app_template.py
```

### 🏆 Points Bonus
- **API FastAPI** : exposer votre modèle en REST
- **Alertes temps réel** : monitoring automatique
- **Déploiement cloud** : Hugging Face Spaces, Streamlit Cloud, GCP

---

### ✅ Checklist de Soumission
- [ ] Notebook documenté & reproductible (seed fixé)
- [ ] Métriques calculées et commentées
- [ ] Dashboard fonctionnel avec lien d'accès
- [ ] Pitch deck PDF (5–7 slides)
- [ ] Vidéo de démo (≤ 3 min)
- [ ] Repository GitHub avec `requirements.txt`

---
<div align='center'>

**Bonne chance ! 🚀🌍**  
*Hackathon IndabaX Cameroon 2026*

</div>